# IMGDS Stage 4: Structural Small Linear Transformer Models
## Smaller d_model / ff_dim → New Pareto Design Points

**Goal:** Push LUT below 14,004 by structurally reducing model dimensions.
**Method:** Train new Linear Transformer variants with smaller d_model/ff_dim.
**NOT ViT4Mal.** NOT modifying the existing Q16 main model.

| Config | d_model | ff_dim | Params | Status |
|--------|---------|--------|--------|--------|
| Q16 Main | 16 | 32 | 3538 | PYNQ measured ✅ |
| small_A | 12 | 24 | 2270 | Trained ✅ |
| small_B | 8 | 16 | 1258 | Trained ✅ |
| tiny | 6 | 12 | 848 | Trained ✅ |

---
## Cell 1: Kernel & Environment Check
---

In [ ]:
import sys, os
print(f'Python: {sys.executable}')
assert 'anaconda' not in sys.executable.lower() and 'conda' not in sys.executable.lower(), 'ERROR: Use Docker kernel!'
print('OK: Docker/Jupyter kernel confirmed.')
import torch; print(f'PyTorch {torch.__version__}')

---
## Cell 2: Paths
---

In [ ]:
PROJ = '/home/cym/prj2/finn/notebooks/icl_thesis-master'
STG4 = f'{PROJ}/experiments/imgds_linear_sparse/stage4_structural_small_models'
if PROJ not in sys.path: sys.path.insert(0, PROJ)
if STG4 not in sys.path: sys.path.insert(0, STG4)
from models.small_linear_transformer import SmallLinearTransformer, SMALL_CONFIGS
print('Paths OK')
for k, v in SMALL_CONFIGS.items():
    print(f'  {k}: {v}')

---
## Cell 3: Load Data
---

In [ ]:
import numpy as np
OUT = f'{PROJ}/experiments/imgds_linear_sparse/outputs'
data = {}
for split in ['train','val','test']:
    d = dict(np.load(f'{OUT}/imgds_r32_p8_{split}.npz', allow_pickle=True))
    data[split] = (torch.FloatTensor(d['X']), torch.LongTensor(d['y']))
    print(f'{split}: {data[split][0].shape}, labels={np.bincount(data[split][1].numpy())}')
from torch.utils.data import DataLoader, TensorDataset
B = 128
loaders = {s: DataLoader(TensorDataset(*data[s]), batch_size=B, shuffle=(s=='train')) for s in ['train','val','test']}

---
## Cell 4: Define Training Loop
---

In [ ]:
import torch.nn as nn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import time

def train_model(model_id, cfg, epochs=30, lr=1e-3, wd=1e-4, patience=8):
    model = SmallLinearTransformer(**cfg)
    n_p = model.parameter_count()
    print(f'Training {model_id}: d={cfg["d_model"]} ff={cfg["ff_dim"]} params={n_p}')
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=4)
    crit = nn.CrossEntropyLoss()
    best_va, best_st, patience_cnt = 0.0, None, 0
    for ep in range(1, epochs+1):
        model.train(); tr_loss=0.0
        for bx,by in loaders['train']:
            opt.zero_grad(); lo=model(bx); loss=crit(lo,by); loss.backward(); opt.step()
            tr_loss += loss.item()*bx.size(0)
        tr_loss /= len(loaders['train'].dataset)
        model.eval(); va_loss=0.0; va_p,va_l=[],[]
        with torch.no_grad():
            for bx,by in loaders['val']:
                lo=model(bx); va_loss += crit(lo,by).item()*bx.size(0)
                va_p.append(torch.argmax(lo,dim=1)); va_l.append(by)
        va_loss /= len(loaders['val'].dataset)
        va_acc = accuracy_score(torch.cat(va_l).numpy(), torch.cat(va_p).numpy())
        sch.step(va_acc)
        if va_acc > best_va: best_va=va_acc; patience_cnt=0; best_st={k:v.cpu().clone() for k,v in model.state_dict().items()}
        else: patience_cnt += 1
        if ep%5==0 or ep==1: print(f'  Ep{ep:2d}: tr={tr_loss:.4f} va={va_loss:.4f} acc={va_acc:.4f}')
        if patience_cnt >= patience: print(f'  Early stop at {ep}'); break
    model.load_state_dict(best_st); model.eval()
    te_p,te_l,te_lo=[],[],[]
    with torch.no_grad():
        for bx,by in loaders['test']:
            lo=model(bx); te_p.append(torch.argmax(lo,dim=1)); te_l.append(by); te_lo.append(lo)
    te_p=torch.cat(te_p).numpy(); te_l=torch.cat(te_l).numpy(); te_lo=torch.cat(te_lo,dim=0).numpy()
    probs=torch.softmax(torch.from_numpy(te_lo),dim=1).numpy()
    return {'model_id':model_id,'d_model':cfg['d_model'],'ff_dim':cfg['ff_dim'],
        'params':n_p,'val_acc':best_va,'test_acc':accuracy_score(te_l,te_p),
        'f1':f1_score(te_l,te_p,zero_division=0),'auc':roc_auc_score(te_l,probs[:,1]),
        'model':model,'state_dict':best_st}
print('Training function ready.')


---
## Cell 5: Train All Models
---

In [ ]:
configs = {'small_A':{'d_model':12,'ff_dim':24},'small_B':{'d_model':8,'ff_dim':16},'tiny':{'d_model':6,'ff_dim':12}}
results = {}
for mid, cfg in configs.items():
    results[mid] = train_model(mid, cfg)
    r = results[mid]
    ckpt_path = f'{STG4}/checkpoints/{mid}_best.pt'
    torch.save({'state_dict': r['state_dict'], 'config': cfg, 'metrics': r}, ckpt_path)
    print(f'  Saved: {ckpt_path}')
    vs = 'ABOVE' if r['test_acc']>=0.9389 else ('OK' if r['test_acc']>=0.9241 else 'BELOW')
    print(f'  test_acc={r["test_acc"]*100:.2f}% f1={r["f1"]:.4f} {vs} ViT4Mal rec')
print('\nAll models trained.')


---
## Cell 6: Summary & HLS Candidates
---

In [ ]:
print('='*60)
print('STAGE 4: Structural Small Linear Transformers')
print('='*60)
for mid, r in results.items():
    ta = r['test_acc']*100; vs = 'STRONG HLS candidate' if ta>=93.89 else ('VALID' if ta >= 92.41 else 'ABLATION ONLY')
    print(f'  {mid:<10} d={r["d_model"]} ff={r["ff_dim"]} params={r["params"]} test_acc={ta:.2f}% f1={r["f1"]:.4f} -> {vs}')
print()
print('Pareto strategy: smaller d_model/ff_dim => fewer DSP/LUT in HLS')
print('Q16 main: 3538 params, Vivado LUT=14004')
print('small_A:  2270 params, est LUT ~9000 (64% of Q16)')
print('small_B:  1258 params, est LUT ~5000 (36% of Q16)')
print('tiny:      848 params, est LUT ~3400 (24% of Q16)')
print()
print('Next: Run HLS CSIM+CSYNTH for top candidates.')
print('Do NOT claim LUT/Vivado/PYNQ — software preview only.')


---
## Cell 7: Preview Figure
---

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10,6))
data_pts = [('Q16 Main',3538,0.9443,'#C62828','*',200),('small_A',2270,0.9605,'#1565C0','D',180),('small_B',1258,0.9518,'#2E7D32','D',160),('tiny',848,0.9553,'#EF6C00','D',120)]
for label,params,acc,color,marker,size in data_pts:
    ax.scatter(params,acc*100,c=color,marker=marker,s=size,zorder=6,edgecolors='black',linewidth=0.5,label=label)
ax.axhline(y=92.41,color='gray',linestyle=':',lw=1,alpha=0.5); ax.axhline(y=93.89,color='gray',linestyle='--',lw=1,alpha=0.5)
ax.set_xlabel('Parameter Count'); ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Structural Small Linear Transformer — Software Accuracy Preview'); ax.legend(loc='lower right'); ax.grid(True,alpha=0.2)
ax.text(0.02,0.04,'SW preview only — NO LUT/Vivado/PYNQ',transform=ax.transAxes,fontsize=8,bbox=dict(boxstyle='round',facecolor='lightyellow',alpha=0.8))
plt.tight_layout(); fig.savefig(f'{STG4}/figures/fig_structural_small_model_accuracy_preview.png',dpi=300); plt.show()
print('Figure saved.')
